In [7]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import combinations

#### Check data format of proteomics data

In [8]:
proteomics_path = Path("../data/raw/proteomics_data.txt")
proteomics_df = pd.read_csv(proteomics_path, sep='\t', index_col=0)
proteomics_df.head(10)
proteomics_df.shape

(12241, 118)

#### Check data format of CORUM datasets.

In [4]:
corum_path = Path("../data/raw/CORUM_data.txt")
corum_df = pd.read_csv(corum_path, sep='\t')
corum_df[corum_df['ComplexID'] == 50]

,ComplexID,ComplexName,Organism,Synonyms,Cell.line,subunits.UniProt.IDs.,subunits.Entrez.IDs.,Protein.complex.purification.method,GO.ID,GO.description,FunCat.ID,FunCat.description,PubMed.ID,Complex.comment,Disease.comment,Subunits.comment,SWISSPROT.organism,subunits.Protein.name.,subunits.Gene.name.,subunits.Gene.name.syn.
18,50,RAB5A-RABEP1-RABGEF complex,Human,RAB5-RABAPTIN5\u2013RABEX5 complex,HeLa cells,P20339;Q15276;Q9UJ41,5868;9135;27342,MI:0018-two hybrid;MI:0019-coimmunoprecipitati...,GO:0006897;GO:0061025;GO:0034058,endocytosis;membrane fusion;endosomal vesicle ...,20.09.18.09.01,endocytosis,9323142,"The Rab5-binding protein, Rabex-5, forms a tig...",NaN,NaN,Homo sapiens (Human);Homo sapiens (Human);Homo...,Ras-related protein Rab-5A;Rab GTPase-binding ...,RAB5A;RABEP1;RABGEF1,"RAB5;RABAPTIN-5, RAB5EP, RABPT5, RABPT5A;RABEX5"


#### I would love to using CORUM as validation. Therefore, I would like to check whether there are proteins in proteomics data can be found in the CORUMS.

In [5]:
# ============================================================
# File paths
# ============================================================
output_dir = Path("../outputs/dataset_overlap")
output_dir.mkdir(exist_ok=True)

# ============================================================
# Helper functions
# ============================================================

def clean_gene_name(gene):
    if pd.isna(gene):
        return None

    gene = str(gene).strip()

    if gene == "" or gene.lower() in {"none", "nan", "na"}:
        return None

    return gene


def parse_corum_gene_list(gene_string):
    if pd.isna(gene_string):
        return []

    genes = str(gene_string).split(";")
    genes = [clean_gene_name(g) for g in genes]
    genes = [g for g in genes if g is not None]

    # Remove duplicates while preserving order
    seen = set()
    unique_genes = []

    for g in genes:
        if g not in seen:
            seen.add(g)
            unique_genes.append(g)

    return unique_genes



#### Though NaN are informative, proteins that do not detected from any patients are less likely to play vital rows in the setting. I would exclude those proteins. There are 99 proteins not detected from any patients.

In [7]:
save_processed_dir = Path("../data/processed")
save_processed_dir.mkdir(exist_ok=True)

# ============================================================
proteomics_df_processed = proteomics_df.dropna(how='all')
proteomics_df_processed.to_csv(save_processed_dir / "proteomics_data_processed.csv", index=True)


In [8]:
# ============================================================
# 1. Drop rows where all elements are NaN
proteomics_df_reversed = proteomics_df.dropna(how='all')

proteomics_df_reversed = 2 ** proteomics_df_reversed
# 3. Fill the remaining NaN values with 0
proteomics_df_reversed = proteomics_df_reversed.fillna(0)
# 4. Save to CSV
proteomics_df_reversed.to_csv(save_processed_dir / "proteomics_data_reversed.csv", index=True)

In [1]:
check_df = pd.read_csv(save_processed_dir / "proteomics_data_processed.csv", index_col=0)
check_df.head()

NameError: name 'pd' is not defined

#### Now I would like to check the overlap in two datasets.

In [5]:

# ============================================================
# 1. Read proteomics data
# ============================================================

print("First 5 protein names:", proteomics_df.index[:5].tolist())

proteomics_genes = [
    clean_gene_name(g)
    for g in proteomics_df.index.tolist()
]

proteomics_genes = [
    g for g in proteomics_genes
    if g is not None
]

proteomics_gene_set = set(proteomics_genes)

print("Number of proteins in proteomics data:", len(proteomics_gene_set))


# Optional: remove proteins with all missing values
non_all_missing_proteins = proteomics_df.index[
    proteomics_df.notna().sum(axis=1) > 0
].tolist()

non_all_missing_gene_set = set(
    clean_gene_name(g)
    for g in non_all_missing_proteins
    if clean_gene_name(g) is not None
)

print("Number of proteins with at least one quantified value:", len(non_all_missing_gene_set))
print("Number of proteins with all missing values:", len(proteomics_gene_set - non_all_missing_gene_set))

# Optional: use only proteins with enough observed patient values
min_observed_patients = 30

well_observed_proteins = proteomics_df.index[
    proteomics_df.notna().sum(axis=1) >= min_observed_patients
].tolist()

well_observed_gene_set = set(
    clean_gene_name(g)
    for g in well_observed_proteins
    if clean_gene_name(g) is not None
)

print(
    f"Number of proteins with at least {min_observed_patients} observed patients:",
    len(well_observed_gene_set)
)
print(
    f"Number of proteins with fewer than {min_observed_patients} observed patients:",
    len(proteomics_gene_set - well_observed_gene_set)
)


First 5 protein names: ['A1BG', 'A1CF', 'A2M', 'A2ML1', 'A4GALT']
Number of proteins in proteomics data: 12241
Number of proteins with at least one quantified value: 12142
Number of proteins with all missing values: 99
Number of proteins with at least 30 observed patients: 10727
Number of proteins with fewer than 30 observed patients: 1514


In [6]:
# ============================================================
# 2. Read CORUM data
# ============================================================

print("\nCORUM shape:", corum_df.shape)
print("CORUM columns:", corum_df.columns.tolist())

# Optional: Keep human complexes - I do this becuase the proteomics data is from human AML patients
if "Organism" in corum_df.columns:
    corum_df = corum_df[
        corum_df["Organism"].astype(str).str.contains("Human", case=False, na=False)
    ].copy()

print("Number of human CORUM complexes:", len(corum_df))

gene_col = "subunits.Gene.name."

if gene_col not in corum_df.columns:
    raise ValueError(f"Cannot find column: {gene_col}")


# ============================================================
# 3. Extract CORUM protein set
# ============================================================

corum_gene_set = set()

for gene_string in corum_df[gene_col]:
    genes = parse_corum_gene_list(gene_string)
    corum_gene_set.update(genes)

print("Number of unique CORUM proteins:", len(corum_gene_set))



CORUM shape: (3627, 20)
CORUM columns: ['ComplexID', 'ComplexName', 'Organism', 'Synonyms', 'Cell.line', 'subunits.UniProt.IDs.', 'subunits.Entrez.IDs.', 'Protein.complex.purification.method', 'GO.ID', 'GO.description', 'FunCat.ID', 'FunCat.description', 'PubMed.ID', 'Complex.comment', 'Disease.comment', 'Subunits.comment', 'SWISSPROT.organism', 'subunits.Protein.name.', 'subunits.Gene.name.', 'subunits.Gene.name.syn.']
Number of human CORUM complexes: 3627
Number of unique CORUM proteins: 4415


In [ ]:
# ============================================================
# 4. Check overlap
# ============================================================

"""
You can choose which proteomics set to use:
1. proteomics_gene_set: all proteins in file
2. non_all_missing_gene_set: proteins with at least one observed value
3. well_observed_gene_set: proteins with enough observed values
"""
selected_proteomics_gene_set = non_all_missing_gene_set

overlap_genes = selected_proteomics_gene_set & corum_gene_set
proteomics_only = selected_proteomics_gene_set - corum_gene_set
corum_only = corum_gene_set - selected_proteomics_gene_set

print("\n========== Protein overlap ==========")
print("Selected proteomics proteins:", len(selected_proteomics_gene_set))
print("CORUM proteins:", len(corum_gene_set))
print("Overlap proteins:", len(overlap_genes))
print("Proteomics-only proteins:", len(proteomics_only))
print("CORUM-only proteins:", len(corum_only))

print("\nOverlap percentage relative to selected proteomics proteins:")
print(round(len(overlap_genes) / len(selected_proteomics_gene_set) * 100, 2))

print("\nOverlap percentage relative to CORUM proteins:")
print(round(len(overlap_genes) / len(corum_gene_set) * 100, 2))


# Save overlap outputs
pd.DataFrame({"gene": sorted(overlap_genes)}).to_csv(
    output_dir / "overlap_proteomics_CORUM_genes.csv",
    index=False
)

pd.DataFrame({"gene": sorted(proteomics_only)}).to_csv(
    output_dir / "proteomics_only_genes.csv",
    index=False
)

pd.DataFrame({"gene": sorted(corum_only)}).to_csv(
    output_dir / "CORUM_only_genes.csv",
    index=False
)


# ============================================================
# 5. Check CORUM complex coverage
# ============================================================

usable_complex_rows = []

for _, row in corum_df.iterrows():
    complex_id = row.get("ComplexID", None)
    complex_name = row.get("ComplexName", None)

    all_genes = parse_corum_gene_list(row[gene_col])
    matched_genes = [
        g for g in all_genes
        if g in selected_proteomics_gene_set
    ]

    usable_complex_rows.append({
        "ComplexID": complex_id,
        "ComplexName": complex_name,
        "n_CORUM_subunits": len(all_genes),
        "n_matched_in_proteomics": len(matched_genes),
        "coverage": len(matched_genes) / len(all_genes) if len(all_genes) > 0 else 0,
        "CORUM_genes": ";".join(all_genes),
        "matched_genes": ";".join(matched_genes),
        "usable_for_pairs": len(matched_genes) >= 2,
    })

usable_complex_df = pd.DataFrame(usable_complex_rows)

print("\n========== CORUM complex coverage ==========")
print("Total human CORUM complexes:", len(usable_complex_df))
print(
    "Complexes with at least 1 matched protein:",
    (usable_complex_df["n_matched_in_proteomics"] >= 1).sum()
)
print(
    "Complexes with at least 2 matched proteins:",
    usable_complex_df["usable_for_pairs"].sum()
)

usable_complex_df.to_csv(
    output_dir / "CORUM_complex_coverage_in_proteomics.csv",    
    index=False
)





========== Protein overlap ==========
Selected proteomics proteins: 12241
CORUM proteins: 4415
Overlap proteins: 3713
Proteomics-only proteins: 8528
CORUM-only proteins: 702

Overlap percentage relative to selected proteomics proteins:
30.33

Overlap percentage relative to CORUM proteins:
84.1

========== CORUM complex coverage ==========
Total human CORUM complexes: 3627
Complexes with at least 1 matched protein: 3477
Complexes with at least 2 matched proteins: 3080


In [8]:
# ============================================================
# 6. Generate CORUM positive pairs present in proteomics - for each CORUM complex, check which proteins are present in proteomics, and generate all pairs of matched proteins in proteomics
# ============================================================

pair_records = {}

for _, row in corum_df.iterrows():
    complex_id = row.get("ComplexID", None)
    complex_name = row.get("ComplexName", None)

    all_genes = parse_corum_gene_list(row[gene_col])

    matched_genes = sorted([
        g for g in all_genes
        if g in selected_proteomics_gene_set
    ])

    if len(matched_genes) < 2:
        continue

    for protein_1, protein_2 in combinations(matched_genes, 2):
        pair = tuple(sorted([protein_1, protein_2]))

        if pair not in pair_records:
            pair_records[pair] = {
                "protein_1": pair[0],
                "protein_2": pair[1],
                "CORUM_label": 1,
                "complex_ids": [],
                "complex_names": [],
            }

        pair_records[pair]["complex_ids"].append(str(complex_id))
        pair_records[pair]["complex_names"].append(str(complex_name))

corum_pairs_df = pd.DataFrame(pair_records.values())

if len(corum_pairs_df) > 0:
    corum_pairs_df["complex_ids"] = corum_pairs_df["complex_ids"].apply(
        lambda x: ";".join(sorted(set(x)))
    )
    corum_pairs_df["complex_names"] = corum_pairs_df["complex_names"].apply(
        lambda x: ";".join(sorted(set(x)))
    )

print("\n========== CORUM-derived positive pairs ==========")
print("Number of unique CORUM positive pairs in selected proteomics proteins:", len(corum_pairs_df))

corum_pairs_df.to_csv(
    output_dir / "CORUM_positive_pairs_in_proteomics.csv",
    index=False
)

# ============================================================
# 7. Display examples
# ============================================================

print("\nExample overlap genes:")
print(sorted(list(overlap_genes))[:20])

print("\nTop usable CORUM complexes by matched subunits:")
print(
    usable_complex_df
    .query("usable_for_pairs == True")
    .sort_values("n_matched_in_proteomics", ascending=False)
    .head(10)
)

print("\nExample CORUM positive pairs:")
print(corum_pairs_df.head(10))


========== CORUM-derived positive pairs ==========
Number of unique CORUM positive pairs in selected proteomics proteins: 40776

Example overlap genes:
['AAAS', 'AATF', 'ABCB1', 'ABCB11', 'ABCC9', 'ABCD4', 'ABCF2', 'ABI1', 'ABI3', 'ABL1', 'ABL2', 'ABRAXAS1', 'ABRAXAS2', 'ACAD8', 'ACAP1', 'ACD', 'ACE', 'ACIN1', 'ACTB', 'ACTBL2']

Top usable CORUM complexes by matched subunits:
      ComplexID                                        ComplexName  \
3345       8391                             Spliceosome, E complex   
3328       8372                             Spliceosome, A complex   
1215       3055                 Nop56p-associated pre-rRNA complex   
560        1181                              C complex spliceosome   
147         320                        55S ribosome, mitochondrial   
139         306                              Ribosome, cytoplasmic   
149         324               39S ribosomal subunit, mitochondrial   
3327       8371                             Spliceosome, B c

## Use of CORUM Data in This Task

Since there is a meaningful overlap between the proteomics dataset and CORUM, CORUM can be used as a biological reference throughout the analysis.

### 1. Evaluating PPI Prediction Methods

In Stage 1, CORUM-derived co-complex pairs provide a benchmark for evaluating different PPI prediction or similarity-scoring methods.

For example, the PA700 complex in CORUM contains 20 proteins, all of which are present in the proteomics dataset. Since these proteins are known to belong to the same complex, we can test whether a given similarity method assigns high scores to the pairwise relationships among them.

This gives us a concrete metric for checking whether the method can recover known co-complex structure from patient-level protein abundance profiles.

Possible evaluation strategies include:

- checking the average similarity score among proteins within a known CORUM complex
- comparing within-complex similarity against randomly sampled protein pairs
- computing precision@K for top-ranked predicted protein pairs
- checking whether known CORUM pairs are enriched among highly ranked predicted interactions

### 2. Evaluating Module Detection

CORUM can also be used to evaluate the module detection stage.

After constructing the predicted protein network and applying community detection, we can check whether proteins from known CORUM complexes are recovered as coherent network modules.

For example, if the PA700 complex contains 20 proteins that are all present in the proteomics data, then a biologically meaningful network should ideally place these proteins close together or assign many of them to the same detected module.

This allows us to evaluate whether the inferred network preserves known protein-complex structure.

Possible checks include:

- whether proteins from the same CORUM complex fall into the same detected module
- whether a detected module is enriched for proteins from a known CORUM complex
- whether known complex members are closer in the network than random protein sets
- whether module detection recovers known sub-complexes or functional protein groups

### 3. Weakly Supervised Learning

CORUM can also support weakly supervised learning.

Some CORUM complexes contain only two proteins. These two-subunit complexes can be treated as relatively high-confidence positive interaction pairs. They can be used as weak labels to train or calibrate a pairwise interaction model.

For each candidate protein pair, we can compute similarity features from the proteomics profiles, such as:

- Pearson correlation
- Spearman correlation
- cosine similarity
- mutual information

Then, a simple classifier can be trained to distinguish CORUM-supported pairs from sampled unknown pairs.

Possible models include:

- logistic regression
- random forest
- gradient boosting
- support vector machine
- small neural network

The output would be a learned interaction score that combines multiple similarity features, instead of relying on a single metric such as Pearson correlation.

However, this should be described as weak supervision, because non-CORUM pairs are not guaranteed to be true negatives. They are better interpreted as unknown pairs.

### 4. Functional Annotation and Biological Interpretation

CORUM can also help with Stage 3, where detected modules need to be functionally interpreted.

Since CORUM complexes correspond to known protein assemblies, overlap between detected modules and CORUM complexes can provide biological clues.

For example, if a detected module is enriched for proteins from a known proteasome, ribosomal, transcriptional repression, mitochondrial, or chromatin-remodelling complex, this can guide the biological interpretation of that module.

This is especially useful when interpreting patient-level module activity. If one group of AML patients shows high activity in a module overlapping with a known CORUM complex, this may suggest differences in biological programs such as protein degradation, metabolism, translation, transcriptional regulation, or stress response.